# 02 — Localizar problemas para a EJA

Objetivo: reescrever uma amostra do GSM8K em português brasileiro, com cenários cotidianos adequados à EJA, mantendo exatamente os valores numéricos e a resposta. Moedas e unidades são recontextualizadas como dados fictícios brasileiros, não convertidos por câmbio ou equivalência física. A validação rejeita números alterados, unidades/moedas estrangeiras e adaptações sinalizadas como inadequadas. Entrada: `data/raw/gsm8k_extraido.jsonl`. Saída: `data/interim/problemas_localizados.jsonl`. Em `MOCK`, os registros são apenas testes técnicos; em `REAL`, usa Groq por uma única função isolada.

In [ ]:
# Opcional: %pip install openai python-dotenv "pydantic>=2.7"
import json
import os
import random
import re
import time
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict

def localizar_projeto() -> Path:
    '''Encontra a pasta de dados a partir do diretório corrente.'''
    for base in (Path.cwd(), *Path.cwd().parents):
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists():
            return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists():
            return base
    raise FileNotFoundError('Configuração compartilhada não encontrada')

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
load_dotenv(ROOT.parent.parent / '.env')
random.seed(CONFIG['seed'])
ENTRADA = ROOT / 'data' / 'raw' / 'gsm8k_extraido.jsonl'
SAIDA = ROOT / 'data' / 'interim' / 'problemas_localizados.jsonl'
VERSAO_LOCALIZACAO = 'pt-br-eja-v2'
CONTEXTOS = ['feira e compras', 'obra e medidas', 'salário e trabalho', 'ônibus e transporte', 'receita e cozinha']
NUMERO = re.compile(r'(?<![A-Za-z])[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?|(?<![A-Za-z])[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?|(?<![A-Za-z])[-+]?\d+(?:[,.]\d+)?')
UNIDADES_ESTRANGEIRAS = re.compile(r'\b(?:miles?|milhas?|pounds?|libras?|lbs?|inches?|polegadas?|feet|foot|p[eé]s|yards?|gallons?|gal[oõ]es|fahrenheit)\b|°F', re.IGNORECASE)
DOLAR_NAO_LOCALIZADO = re.compile(r'(?<!R)\$')

class Localizacao(BaseModel):
    '''Campos obrigatórios retornados pelo adaptador de linguagem.'''
    model_config = ConfigDict(extra='forbid')
    problema_localizado: str
    contexto_cotidiano: str
    tema: str

class RegistroLocalizado(BaseModel):
    '''Valida campos essenciais e preserva os campos de origem GSM8K.'''
    model_config = ConfigDict(extra='allow')
    id: str
    base_id: str
    problema_localizado: str
    contexto_cotidiano: str
    resposta_numerica: str
    modo_geracao: str
    localizacao_versao: str

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Preserva JSONL de uma linha mesmo com separadores Unicode no texto.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

def arquivar_saida_antiga(caminho: Path) -> Path:
    '''Move uma saída incompatível para um backup único, sem sobrescrever arquivos.'''
    marca_tempo = time.strftime('%Y%m%d_%H%M%S')
    backup = caminho.with_name(f'{caminho.stem}_versao_anterior_{marca_tempo}{caminho.suffix}')
    indice = 1
    while backup.exists():
        backup = caminho.with_name(f'{caminho.stem}_versao_anterior_{marca_tempo}_{indice}{caminho.suffix}')
        indice += 1
    caminho.replace(backup)
    return backup

def numero_decimal(texto: str) -> Decimal:
    '''Lê número inteiro ou decimal simples sem executar código.'''
    limpo = texto.strip()
    if re.fullmatch(r'[-+]?\d{1,3}(,\d{3})+(\.\d+)?', limpo):
        limpo = limpo.replace(',', '')
    elif re.fullmatch(r'[-+]?\d{1,3}(\.\d{3})+(,\d+)?', limpo):
        limpo = limpo.replace('.', '').replace(',', '.')
    else:
        limpo = limpo.replace(',', '.')
    try:
        return Decimal(limpo)
    except InvalidOperation as erro:
        raise ValueError(f'Número inválido: {texto}') from erro

def numeros(texto: str) -> Counter[Decimal]:
    '''Conta os valores numéricos literais de um texto.'''
    return Counter(numero_decimal(m.group()) for m in NUMERO.finditer(texto))

def validar_localizacao(origem: dict[str, Any], localizado: dict[str, str]) -> None:
    '''Rejeita números alterados, unidades estrangeiras e dólares não reescritos em reais.'''
    problema = localizado['problema_localizado']
    if problema.strip().upper().startswith('DESCARTAR:'):
        raise ValueError(problema.strip())
    if numeros(origem['pergunta_original']) != numeros(problema):
        raise ValueError('A localização alterou, omitiu ou acrescentou números')
    if '$' in origem['pergunta_original'] and 'R$' not in problema:
        raise ValueError('Valores em dólar devem ser reescritos em reais (R$)')
    if DOLAR_NAO_LOCALIZADO.search(problema):
        raise ValueError('Marcador de dólar não localizado; use R$')
    unidade = UNIDADES_ESTRANGEIRAS.search(problema)
    if unidade:
        raise ValueError(f'Unidade estrangeira não localizada: {unidade.group()}')

def resposta_numerica(texto: str) -> Decimal:
    '''Extrai o último valor numérico após o marcador final do GSM8K.'''
    achados = [m.group() for m in NUMERO.finditer(texto)]
    if not achados:
        raise ValueError('Resposta final sem número reconhecível')
    return numero_decimal(achados[-1])

def chamar_llm(pergunta: str, contexto: str) -> dict[str, str]:
    '''Adapta o enunciado; não calcula nem decide respostas matemáticas.'''
    if CONFIG['llm_mode'].upper() == 'MOCK':
        return Localizacao.model_validate({'problema_localizado': f'Exemplo técnico MOCK (não usar para treino). Texto de origem: {pergunta}',
                'contexto_cotidiano': contexto, 'tema': 'teste de pipeline'}).model_dump()
    if CONFIG['llm_mode'].upper() != 'REAL':
        raise ValueError('llm_mode deve ser MOCK ou REAL')
    from openai import OpenAI, RateLimitError
    api_key = os.getenv('GROQ_API_KEY')
    if not api_key:
        raise RuntimeError('Defina GROQ_API_KEY no ambiente ou no .env da raiz do projeto')
    cliente = OpenAI(base_url=CONFIG['llm_base_url'], api_key=api_key, max_retries=0)
    instrucao = ('Reescreva, não traduza literalmente, em português brasileiro simples e respeitoso para adultos da EJA. '
        'Use cenário plausível do cotidiano de adultos no Brasil (trabalho, feira, transporte, obra, casa ou receita) '
        'e nomes fictícios brasileiros; evite personagens, lugares e costumes dos EUA. '
        'Use frases curtas, reais (R$) e unidades métricas comuns no Brasil. '
        'Mantenha exatamente os mesmos valores numéricos, inclusive repetições, e a mesma relação matemática. '
        'Ao trocar dólares por reais ou unidades estrangeiras por unidades locais, trate os valores mantidos como dados '
        'fictícios de um novo exercício; não afirme que fez conversão cambial ou física. '
        'Não invente nem retire números, não calcule nem revele a resposta. '
        'Se não houver adaptação brasileira natural sem mudar os números, coloque DESCARTAR: motivo em problema_localizado. '
        'Retorne somente JSON com problema_localizado, contexto_cotidiano e tema.')
    mensagem_usuario = (f'Situação sugerida, somente se combinar com o problema: {contexto}. '
                        f'Enunciado a reescrever:\n{pergunta}')
    for tentativa in range(7):
        try:
            retorno = cliente.chat.completions.create(model=CONFIG['llm_model'], temperature=0,
                max_tokens=160, response_format={'type': 'json_object'},
                messages=[{'role': 'system', 'content': instrucao}, {'role': 'user', 'content': mensagem_usuario}])
            break
        except RateLimitError as erro:
            if tentativa == 6:
                raise
            cabecalhos = getattr(getattr(erro, 'response', None), 'headers', {})
            try:
                espera_indicada = float(cabecalhos.get('retry-after', 0))
            except (TypeError, ValueError):
                espera_indicada = 0
            espera = max(espera_indicada, min(2 ** tentativa, 30))
            print(f'Limite da Groq; tentando novamente em {espera:.1f}s ({tentativa + 1}/7).')
            time.sleep(espera)
    conteudo = retorno.choices[0].message.content
    if not conteudo:
        raise ValueError('O LLM retornou conteúdo vazio')
    resultado = json.loads(conteudo)
    for campo in ('problema_localizado', 'contexto_cotidiano', 'tema'):
        if not isinstance(resultado.get(campo), str) or not resultado[campo].strip():
            raise ValueError(f'Campo inválido na saída do LLM: {campo}')
    return Localizacao.model_validate(resultado).model_dump()

if not ENTRADA.exists():
    raise FileNotFoundError(f'Execute primeiro o notebook 01: {ENTRADA}')
originais = [json.loads(l) for l in ENTRADA.read_text(encoding='utf-8').splitlines() if l]
amostra = originais[:int(CONFIG['max_examples'])]
SAIDA.parent.mkdir(parents=True, exist_ok=True)
ids_amostra = {origem['id'] for origem in amostra}
localizados_existentes = {}
if SAIDA.exists():
    for numero_linha, linha in enumerate(SAIDA.read_text(encoding='utf-8').splitlines(), start=1):
        if not linha.strip():
            continue
        try:
            registro_existente = json.loads(linha)
        except json.JSONDecodeError as erro:
            raise ValueError(f'JSONL parcial inválido na linha {numero_linha}; preserve uma cópia antes de corrigir') from erro
        if (registro_existente.get('modo_geracao') != CONFIG['llm_mode'].upper()
                or registro_existente.get('localizacao_versao') != VERSAO_LOCALIZACAO):
            backup = arquivar_saida_antiga(SAIDA)
            localizados_existentes.clear()
            print(f'Arquivo anterior preservado em: {backup}; começando nova localização.')
            break
        localizados_existentes[registro_existente['id']] = registro_existente
gerados = retomados = len(localizados_existentes)
descartados = 0
with SAIDA.open('a', encoding='utf-8') as arquivo:
    for origem in amostra:
        if origem['id'] in localizados_existentes:
            continue
        contexto = random.choice(CONTEXTOS)
        try:
            localizado = chamar_llm(origem['pergunta_original'], contexto)
            validar_localizacao(origem, localizado)
            resposta_calculada = resposta_numerica(origem['resposta_final'])
            registro = {**origem, **localizado, 'contexto_cotidiano': localizado.get('contexto_cotidiano', contexto),
                        'resposta_numerica': str(resposta_calculada),
                        'modo_geracao': CONFIG['llm_mode'].upper(),
                        'localizacao_versao': VERSAO_LOCALIZACAO}
            registro = RegistroLocalizado.model_validate(registro).model_dump()
            arquivo.write(serializar_jsonl(registro) + '\n')
            gerados += 1
            localizados_existentes[origem['id']] = registro
        except (ValueError, KeyError, json.JSONDecodeError) as erro:
            descartados += 1
            print(f"Descartado {origem.get('id')}: {erro}")
print(f'Amostra: {len(amostra)} | retomados: {retomados} | gerados: {gerados} | descartados nesta execução: {descartados} | modo: {CONFIG["llm_mode"]}')

A localização troca cenário, nomes, moedas e unidades, mas mantém os valores numéricos como dados fictícios de um novo problema — não como conversão de dólares ou milhas. A validação rejeita números modificados, dólares sem `R$`, unidades estrangeiras e casos que o LLM sinalize como inadequados. A resposta continua sendo conferida com o valor de referência extraído do GSM8K.

In [ ]:
localizados = [json.loads(l) for l in SAIDA.read_text(encoding='utf-8').splitlines() if l]
assert len(localizados) == gerados
assert all(r['modo_geracao'] in ('MOCK', 'REAL') and r['problema_localizado'] for r in localizados)
assert all(r.get('localizacao_versao') == VERSAO_LOCALIZACAO for r in localizados)
for registro in localizados:
    validar_localizacao(registro, {'problema_localizado': registro['problema_localizado'],
        'contexto_cotidiano': registro['contexto_cotidiano'], 'tema': registro['tema']})
print('Registros:', len(localizados), '| modo:', CONFIG['llm_mode'], '| descartados:', descartados)
if localizados:
    amostra_verificacao = localizados[0]
    assert Decimal(amostra_verificacao['resposta_numerica']) == resposta_numerica(amostra_verificacao['resposta_final'])
    print(json.dumps({k: amostra_verificacao[k] for k in ('id', 'problema_localizado', 'contexto_cotidiano', 'resposta_numerica', 'modo_geracao')}, ensure_ascii=False, indent=2))